# CP201A Lab 5: Testing for Statistical Significance

**Fall 2026** (September 30 and October 2)

In Lab 4 you built a neighborhood out of census tracts and gave every estimate a margin of
error. You also saw that the neighborhood's margins of error are several times the city's.
Today we settle the question that raises: whether the gap between the neighborhood's number
and the city's is larger than the margins of error allow. That takes a test rather than a
glance, and by the end of lab you will have run it on your own numbers: your neighborhood
against the city or county you chose to compare it with.

By the end of lab you will have, for your own neighborhood:

* the two significance tests P/NP #5 Part III asks for, with the Z value and the confidence
  level for each
* a table of Z values for every race and ethnicity share, ready for Assignment 1
* one change-over-time test (2015 to 2019 versus 2020 to 2024) on the learning path you chose
* a dollar figure adjusted for inflation, with its margin of error adjusted too

## Learning objectives

**Everyone**
* Pull one value out of a table with `.loc`
* Convert a margin of error to a standard error
* Test whether two ACS estimates are different, first by hand and then with a function
* Choose a confidence level and say what it costs
* Write about a result, including a result that is not significant
* Run the same test across two ACS periods, and adjust dollars before comparing them

**If you want more**
* Draw a bar chart with error bars (a preview of Lab 6)
* Crosswalk 2019 tract data onto 2020 tracts (the companion notebook)

**Reading:** U.S. Census Bureau (2020), *Understanding and Using American Community Survey
Data: What All Data Users Need to Know*, Section 7, Understanding Error and Determining
Statistical Significance (pp. 53 to 58).
https://www.census.gov/programs-surveys/acs/library/handbooks/general.html

**Bring:** the CSV files Lab 4 saved (`lab4_<neighborhood>_comparison_wide.csv` and the
tenure or poverty comparison file from Section 10). Section 1 says where to put them.

**What you submit this week:** nothing from this notebook. P/NP #5 is the filled-in template,
saved as a PDF, on bCourses by Sunday, October 4. Section 5.3 prints the numbers it asks for.

## 0. Before we begin

Same start as Labs 3 and 4. You need the Census API key today only for Section 6 (the second
ACS period) and for the backup pull in Section 1.3, but load it now so it is ready.

In [ ]:
%pip install -q census

In [ ]:
from census import Census
import pandas as pd
import numpy as np
import os

In [ ]:
# Your key was saved to a file in Lab 3. This cell reads it; you do not paste it again.
try:
    with open(os.path.expanduser('~/census_key.txt')) as f:
        api_key = f.read().strip()
    print('Key loaded. It starts with:', api_key[:4] + '...')
except FileNotFoundError:
    print('No key file found. Open Lab 3, run the cell in Section 0.1 once to save your key, then run this cell again.')

c = Census(key=api_key)

### 0.1 The Lab 4 toolkit

The cell below is copied from Lab 4 without changes: the B03002 dictionary, the tenure,
poverty, and median rent dictionaries, and the four functions (`clean_acs`, `aggregate_tracts`,
`add_shares`, `pull_geo`). You read and used all of it last week, so run it and move on. It is
here so that Section 6 can pull the 2015 to 2019 period and so that Section 1.3 can rebuild the
Lab 4 tables if you do not have your CSV files with you.

In [ ]:
# ---- Copied from Lab 4. Nothing new here. ----

variables_of_interest = {
    'NAME': 'NAME',
    'GEO_ID': 'GEO_ID',
    'B03002_001E': 'total',
    'B03002_001M': 'total_moe',
    'B03002_003E': 'nh_white',
    'B03002_003M': 'nh_white_moe',
    'B03002_004E': 'nh_black',
    'B03002_004M': 'nh_black_moe',
    'B03002_005E': 'nh_native',
    'B03002_005M': 'nh_native_moe',
    'B03002_006E': 'nh_asian',
    'B03002_006M': 'nh_asian_moe',
    'B03002_007E': 'nh_pi',
    'B03002_007M': 'nh_pi_moe',
    'B03002_008E': 'nh_1other',
    'B03002_008M': 'nh_1other_moe',
    'B03002_009E': 'nh_multi',
    'B03002_009M': 'nh_multi_moe',
    'B03002_012E': 'hispanic',
    'B03002_012M': 'hispanic_moe',
}
GROUPS = ['hispanic', 'nh_white', 'nh_black', 'nh_native', 'nh_asian', 'nh_pi', 'nh_1other', 'nh_multi']
LABELS = {
    'hispanic':  'Hispanic or Latino (any race)',
    'nh_white':  'White alone, not Hispanic',
    'nh_black':  'Black or African American alone, not Hispanic',
    'nh_native': 'American Indian and Alaska Native alone, not Hispanic',
    'nh_asian':  'Asian alone, not Hispanic',
    'nh_pi':     'Native Hawaiian and Other Pacific Islander alone, not Hispanic',
    'nh_1other': 'Some other race alone, not Hispanic',
    'nh_multi':  'Two or more races, not Hispanic',
    'total':     'Total population',
}

# Question 2, Option B: tenure (B25003). Universe: occupied housing units.
tenure_vars = {
    'NAME': 'NAME', 'GEO_ID': 'GEO_ID',
    'B25003_001E': 'units', 'B25003_001M': 'units_moe',
    'B25003_002E': 'owner', 'B25003_002M': 'owner_moe',
    'B25003_003E': 'renter', 'B25003_003M': 'renter_moe',
}
# Question 2, Option A: poverty (B17001). Universe: population for whom poverty status is determined.
poverty_vars = {
    'NAME': 'NAME', 'GEO_ID': 'GEO_ID',
    'B17001_001E': 'pov_universe', 'B17001_001M': 'pov_universe_moe',
    'B17001_002E': 'below_poverty', 'B17001_002M': 'below_poverty_moe',
}
# Median gross rent (B25064), used in Section 6.3
rent_vars = {
    'NAME': 'NAME', 'GEO_ID': 'GEO_ID',
    'B25064_001E': 'med_rent', 'B25064_001M': 'med_rent_moe',
}


def clean_acs(df, id_cols=('NAME', 'GEO_ID', 'state', 'county', 'tract', 'place')):
    '''Convert estimate and MOE columns to numbers and replace jam values.'''
    df = df.copy()
    numeric_cols = [col for col in df.columns if col not in id_cols]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col])
    moe_cols = [col for col in numeric_cols if col.endswith('_moe')]
    df[moe_cols] = df[moe_cols].replace(-555555555, 0)                       # controlled: MOE is zero
    df[numeric_cols] = df[numeric_cols].replace([-666666666, -222222222, -333333333], np.nan)  # no estimate
    return df


def aggregate_tracts(df, name):
    '''Add up a DataFrame of tracts into one neighborhood row (sum estimates, RSS the MOEs).'''
    numeric_cols = df.select_dtypes('number').columns
    moe_cols = [col for col in numeric_cols if col.endswith('_moe')]
    est_cols = [col for col in numeric_cols if col not in moe_cols]
    estimates = df[est_cols].sum()
    moes = (df[moe_cols]**2).sum()**0.5
    row = pd.concat([estimates, moes])
    out = pd.DataFrame(row).transpose()
    out.insert(0, 'NAME', name)
    out.insert(1, 'n_tracts', len(df))
    return out


def add_shares(df, groups, total='total'):
    '''Add pct_<group> and pct_<group>_moe (in percent) for each group, as a share of total.'''
    df = df.copy()
    y = df[total]
    moe_y = df[f'{total}_moe']
    for g in groups:
        p = df[g] / y
        under_root = df[f'{g}_moe']**2 - p**2 * moe_y**2
        under_root = np.where(under_root < 0, df[f'{g}_moe']**2 + p**2 * moe_y**2, under_root)
        df[f'pct_{g}'] = p * 100
        df[f'pct_{g}_moe'] = under_root**0.5 / y * 100
    return df


def pull_geo(geo_for, geo_in, variables=variables_of_interest, year=2024):
    '''Pull one geography for one year and clean it.'''
    df = pd.DataFrame(
        c.acs5.get(list(variables.keys()), {'for': geo_for, 'in': geo_in}, year=year)
    ).rename(columns=variables)
    return clean_acs(df)

print('Toolkit loaded.')

### 0.2 Picking one value out of a table with `.loc`

Today's test needs four numbers: two estimates and two standard errors, one pair for the
neighborhood and one for the city. They sit in a table, and we need to get each one out by
itself. `.loc` (short for location) does that. It selects by **row label and column name**:

```python
table.loc[row_label, column_name]
```

The row label is whatever the table's index holds. Row numbers (0, 1, 2) are not much use,
so the first move is usually to make a meaningful column the index. A small example with
flowers, alongside the same moves in Excel.

In [ ]:
flowers = pd.DataFrame({'Type': ['Orchid', 'Rose', 'Carnation', 'Daffodil'],
                        'Count': [1, 15, 4, 21],
                        'Color': ['Red', 'White', 'Orange', 'Yellow']})
flowers

**Columns.** `flowers.loc[:, ['Type', 'Color']]` takes every row (`:` means all) for two
columns, which is cells A1 through C5 minus column B in Excel terms. `flowers[['Type', 'Color']]`
is the short form and does the same thing.

<img src="excel_example_a.png" width="300">

**One row.** Set a meaningful index first, then select by its label.

<img src="excel_example_b.png" width="300">

In [ ]:
flowers = flowers.set_index('Type')     # the Type column becomes the row labels
flowers.loc['Orchid']                   # one row, all columns

In [ ]:
flowers.loc['Orchid', 'Count']          # one cell: row label, column name

That last line returns a single number, which is exactly what we want to feed into a formula.

**Rows that meet a condition.** `.loc` also takes a true/false test, like Excel's filter.
`==` (two equals signs) asks whether two things are equal; a single `=` assigns. Each
condition goes in its own parentheses, `&` means and, `|` means or.

<img src="excel_example_c.png" width="400">

In [ ]:
flowers.loc[flowers['Count'] > 10]

In [ ]:
flowers.loc[(flowers['Count'] > 10) & (flowers['Color'] == 'Yellow')]

In [ ]:
# EXERCISE #1: two lines. First, use .loc to pull out the color of the Carnation (one cell).
# Second, use .loc to show the rows where Count is less than 5 or more than 20.

## 1. Your data from Lab 4

### 1.1 Where the files are

Lab 4 saved its CSVs in the Week 5 folder of your Datahub account
(`PlanningMethods2026/Week05_Lab_AggregatingMOEs/`). This notebook is in the Week 6 folder next
to it, so the default path below reaches over with `../`. If the files are still there, you
do not need to do anything.

If they are not (you re-pulled Lab 4, or you are working from the backup copies you downloaded
to your computer), upload them to Datahub. In the Datahub file browser, open the folder you
want, click the upload button (the up arrow in the toolbar above the file list), choose the
CSVs on your computer, and confirm. Then tell the notebook where they are:

* Uploaded to the **Week 5** folder (`Week05_Lab_AggregatingMOEs`): leave `LAB4_FOLDER` as it
  is below.
* Uploaded to the **Week 6** folder (this one): set `LAB4_FOLDER = ''`.

Do not rename the files when you upload them; the notebook builds the file name from
`NEIGHBORHOOD_NAME`, so `lab4_west_oakland_comparison_wide.csv` must still be called that.

Two files are needed: the race and ethnicity comparison (`_comparison_wide.csv`) and the file
for the Question 2 option you chose (`_tenure_comparison_wide.csv` or
`_poverty_comparison_wide.csv`). Set the names below to match your neighborhood; the default
is West Oakland with tenure.

**If your Lab 4 CSVs came from the West Oakland defaults, they are not your data.** It is
fine to run today's lab on them to learn the test. Before P/NP #5, open your Lab 4 notebook,
put your own tract list in Section 1.1, run the whole notebook again (Section 10 too), and
use the CSVs it saves. From here on, everything you hand in (P/NP #5 and both parts of
Assignment 1) is built from your own neighborhood's tracts: your 2020 list for 2020 to 2024,
and the 2010 list that covers the same land for 2015 to 2019. Section 6.0 tells you what
that 2010 list is.

In [ ]:
NEIGHBORHOOD_NAME = 'West Oakland'
LAB4_FOLDER = '../Week05_Lab_AggregatingMOEs/'
Q2_OPTION = 'tenure'          # 'tenure' (Option B) or 'poverty' (Option A)

safe_name = NEIGHBORHOOD_NAME.lower().replace(' ', '_')
RACE_FILE = f'{LAB4_FOLDER}lab4_{safe_name}_comparison_wide.csv'
Q2_FILE = f'{LAB4_FOLDER}lab4_{safe_name}_{Q2_OPTION}_comparison_wide.csv'

# The column that holds the share we will test in Question 2, and its universe
Q2_SHARE = {'tenure': 'pct_renter', 'poverty': 'pct_below_poverty'}[Q2_OPTION]

### 1.2 Read the two tables and set the index

Each file has three rows: the neighborhood, the city, and the county. We make `NAME` the
index so that `.loc` can find rows by name. Run the cell and read the printed index: those are
the exact labels you will use in Section 3.

In [ ]:
race = pd.read_csv(RACE_FILE).set_index('NAME')
q2 = pd.read_csv(Q2_FILE).set_index('NAME')

print('Rows in your tables (copy these labels exactly):')
for label in race.index:
    print('   ', repr(label))

race[['total', 'total_moe', 'pct_hispanic', 'pct_hispanic_moe', 'pct_nh_black', 'pct_nh_black_moe']].round(1)

Now name your comparison. Part I of the assignment compares the neighborhood with **either**
the city or the county; pick the one you chose in P/NP #5 and paste its label from the list
above. Everything after this uses these two names.

In [ ]:
NBHD = NEIGHBORHOOD_NAME               # the neighborhood row
COMP = 'Oakland city, California'      # your comparison geography: paste the label printed above

q2.loc[[NBHD, COMP]].round(1)

### 1.3 If you do not have the files

Skip this cell if Section 1.2 ran. If it raised `FileNotFoundError`, this cell rebuilds both
tables from the API for the West Oakland default (the same 13 tracts as Lab 4) so you can do
the lab. Before P/NP #5, go back to your Lab 4 notebook, run it with your own tracts, and save
the CSVs; the test is the same, only the numbers change.

In [ ]:
# Backup only. Rebuilds the two Lab 4 comparison tables for the West Oakland default.
# Remove the # from the last line to run it.
def rebuild_lab4_tables(tract_list, state='06', county='001', place='53000', name='West Oakland'):
    def chain(variables, groups, total):
        alltr = pull_geo('tract:*', f'state:{state} county:{county}', variables)
        tracts = alltr[alltr['tract'].isin(tract_list)].copy()
        nbhd = add_shares(aggregate_tracts(tracts, name), groups, total=total)
        city = add_shares(pull_geo(f'place:{place}', f'state:{state}', variables), groups, total=total)
        county_row = add_shares(pull_geo(f'county:{county}', f'state:{state}', variables), groups, total=total)
        return pd.concat([nbhd, city, county_row], ignore_index=True).set_index('NAME')
    race = chain(variables_of_interest, GROUPS, 'total')
    if Q2_OPTION == 'tenure':
        q2 = chain(tenure_vars, ['owner', 'renter'], 'units')
    else:
        q2 = chain(poverty_vars, ['below_poverty'], 'pov_universe')
    return race, q2

WEST_OAKLAND_13 = ['401400', '401500', '401600', '401700', '401800', '402200', '402400',
                   '402500', '402600', '402700', '410500', '981900', '982000']
# race, q2 = rebuild_lab4_tables(WEST_OAKLAND_13)

## 2. From margin of error to standard error

Every ACS margin of error is published at the **90 percent confidence level**. That is a
choice the Census Bureau made, and it means each MOE is 1.645 standard errors wide:

$$MOE_{90} = 1.645 \times SE \qquad\text{so}\qquad SE = \frac{MOE_{90}}{1.645}$$

The standard error is the quantity the test needs, because it is the one that adds up across
two estimates (Section 3). The conversion is one division. Here it is for the renter share
of the neighborhood, using `.loc` from Section 0.2. (If you chose poverty, `Q2_SHARE` already
points at the poverty rate; the code is the same.)

In [ ]:
moe = q2.loc[NBHD, f'{Q2_SHARE}_moe']
se = moe / 1.645
print(f'{NBHD}: {Q2_SHARE} MOE = {moe:.2f} percentage points, SE = {se:.2f}')

### 2.1 Every margin of error at once

Our tables have many MOE columns, and every one of them ends in `_moe` because we were
careful with names in Lab 4. So we can loop over the column names, and for each one that ends
in `_moe`, make a matching `_se` column. `col.endswith('_moe')` asks the question; `col[:-4]`
is the column name with its last four characters (`_moe`) cut off, so `pct_renter_moe` becomes
`pct_renter` and we add `_se` to it.

In [ ]:
for col in q2.columns:
    if col.endswith('_moe'):
        q2[col[:-4] + '_se'] = q2[col] / 1.645

q2[[Q2_SHARE, f'{Q2_SHARE}_moe', f'{Q2_SHARE}_se']].round(2)

Wrapped in a function, so we can do the same to any table in one line. (The loop above
already changed `q2`; running the function on it again is harmless, since it only recomputes
the same columns.)

In [ ]:
def add_standard_errors(df):
    '''Add a <col>_se column (MOE / 1.645) for every <col>_moe column in df.

    Input: a DataFrame with margin-of-error columns whose names end in _moe (ACS 90 percent MOEs)
    Output: a copy of df with a matching _se column for each one
    '''
    df = df.copy()
    for col in df.columns:
        if col.endswith('_moe'):
            df[col[:-4] + '_se'] = df[col] / 1.645
    return df

q2 = add_standard_errors(q2)

In [ ]:
# EXERCISE #2: run add_standard_errors on the race table and store the result back in race.
# Then show, for the neighborhood and the comparison rows only, the estimate, MOE, and SE
# for pct_hispanic. Tip: race.loc[[NBHD, COMP], [three column names]]

## 3. The test, by hand

### 3.1 What the test asks

Two estimates, one for the neighborhood and one for the city, are almost never exactly equal.
The question is whether the gap between them is bigger than the noise in the two estimates.
The handbook's test for two ACS estimates (Section 7, pp. 55 to 56) is:

$$Z = \frac{\left|\hat{X}_1 - \hat{X}_2\right|}{\sqrt{SE_1^2 + SE_2^2}}$$

Read it as: the gap, divided by the combined uncertainty. The numerator is the absolute value
of the difference (the vertical bars), so the order of the two estimates does not matter. The
denominator combines the two standard errors by root sum of squares, the same move as adding
tract MOEs in Lab 4 and for the same reason: errors in two independent estimates do not all
point the same way.

The result, $Z$, is the gap measured in units of its own uncertainty. A $Z$ of 1 means the gap
is about as big as the noise; a $Z$ of 3 means the gap is three times the noise. The formal
version: the **null hypothesis** is that the two places have the same true value, so the true
difference is zero. If $Z$ is large, the data are hard to square with that hypothesis, and we
**reject** it and call the difference **statistically significant**. If $Z$ is small, we
**fail to reject** it: the data cannot tell the difference from zero.

In Python we have everything: subtraction, `abs()` for the absolute value, `**2` to square,
`**0.5` for the square root. Four `.loc` lookups and one line of arithmetic.

In [ ]:
# The four ingredients, pulled out by name
x1  = q2.loc[NBHD, Q2_SHARE]
x2  = q2.loc[COMP, Q2_SHARE]
se1 = q2.loc[NBHD, f'{Q2_SHARE}_se']
se2 = q2.loc[COMP, f'{Q2_SHARE}_se']

print(f'{NBHD}: {x1:.1f} percent (SE {se1:.2f})')
print(f'{COMP}: {x2:.1f} percent (SE {se2:.2f})')
print(f'Gap: {abs(x1 - x2):.1f} percentage points')

In [ ]:
# The test
z = abs(x1 - x2) / (se1**2 + se2**2)**0.5
print(f'Z = {z:.2f}')

### 3.2 Critical values, and what they cost

$Z$ is compared with a **critical value** that depends on how confident you want to be:

| Confidence level | Critical value | A 90 percent MOE becomes this level's MOE when multiplied by |
|---|---|---|
| 90 percent | 1.645 | 1.000 |
| 95 percent | 1.960 | 1.960 / 1.645 = 1.192 |
| 99 percent | 2.576 | 2.576 / 1.645 = 1.566 |

If $Z$ is larger than 1.645, the difference is significant at the 90 percent level. If it is
also larger than 1.960, at 95 percent; if larger than 2.576, at 99 percent. The figure shows
the same thing: the shaded tails are where $Z$ has to land.

<img src="z_critical_values.png" width="640">

The third column is the tradeoff in a number. Asking for more confidence means a wider
interval around every estimate (multiply the published MOE by 1.192 for 95 percent, 1.566 for
99 percent), and a wider interval means a harder test to pass. A difference that is
significant at 90 percent may not be at 95. There is no free lunch here: you can be more sure
about a vaguer claim, or less sure about a sharper one. The ACS handbook uses 90 percent
because that is the level the MOEs are published at, and it is fine to use in this course.
Whatever you choose, name it in the note under the exhibit, and use the same level
throughout an exhibit.

(Method: U.S. Census Bureau, 2020, Section 7, pp. 55 to 56, and Section 8 for the MOE
conversion.)

In [ ]:
if z > 2.576:
    level = 'significant at the 99 percent confidence level'
elif z > 1.960:
    level = 'significant at the 95 percent confidence level'
elif z > 1.645:
    level = 'significant at the 90 percent confidence level'
else:
    level = 'not statistically significant at the 90 percent confidence level'

print(f'Z = {z:.2f}: the difference is {level}.')

### 3.3 Saying it in a sentence

The sentence a reader needs has four parts: the two estimates with their margins of error,
the direction of the difference, the confidence level, and the $Z$ value. For example:

> The renter share in West Oakland (xx.x percent, MOE y.y) is higher than in Oakland as a
> whole (xx.x percent, MOE y.y), and the difference is statistically significant at the
> 95 percent confidence level (Z = 2.3).

Report $Z$ and the level. If you also report a p value, get its direction right: a
significant result has a **small** p (p < .05), not a large one. Writing p > .05 when you mean
p < .05 reverses your claim, and it is one of the most common mistakes in past memos.

The cell below writes the sentence from the numbers, so the numbers in your text cannot drift
away from the numbers in your table.

In [ ]:
direction = 'higher' if x1 > x2 else 'lower'
print(f'The {Q2_SHARE} in {NBHD} ({x1:.1f} percent, MOE {q2.loc[NBHD, f"{Q2_SHARE}_moe"]:.1f}) is {direction} '
      f'than in {COMP} ({x2:.1f} percent, MOE {q2.loc[COMP, f"{Q2_SHARE}_moe"]:.1f}), '
      f'and the difference is {level} (Z = {z:.2f}).')

In [ ]:
# EXERCISE #3: repeat Sections 3.1 to 3.3 by hand for ONE race or ethnicity share from the
# race table (P/NP #5, Comparison 1). Choose the category you plan to write about and say why
# in a comment. Pull the four ingredients with .loc, compute Z, state the level, and print the
# sentence. Use the race table (which now has _se columns from Exercise #2).

## 4. The test as a function

You will run this test many times: two for P/NP #5, one for every claim of a difference in
Assignment 1, and every claim of change in Part II. So we write it once. The function takes
the table, the column to compare, and the two row labels, and returns $Z$. It is Section 3.1
with the four `.loc` lookups inside.

In [ ]:
def z_statistic(df, col, place_1, place_2):
    '''Two-sample Z for the difference between two ACS estimates.

    Inputs:
    - df: a DataFrame indexed by geography (or period) name, with col and col + '_se' columns
    - col: the column to compare (an estimate, usually a pct_ column)
    - place_1, place_2: the two row labels to compare

    Output: Z, the absolute difference divided by the root sum of squares of the two SEs.
    '''
    x1 = df.loc[place_1, col]
    x2 = df.loc[place_2, col]
    se1 = df.loc[place_1, col + '_se']
    se2 = df.loc[place_2, col + '_se']
    return abs(x1 - x2) / (se1**2 + se2**2)**0.5


def significance(z):
    '''Name the highest confidence level a Z value clears (90, 95, or 99 percent), or "not significant".'''
    if z > 2.576:
        return '99 percent'
    elif z > 1.960:
        return '95 percent'
    elif z > 1.645:
        return '90 percent'
    else:
        return 'not significant at 90 percent'


# Same test as Section 3, one line. The number should match.
z_statistic(q2, Q2_SHARE, NBHD, COMP)

### 4.1 Every share at once

With a function, testing all eight race and ethnicity shares is a loop. The result is a
table you can drop straight into Assignment 1 Part I, Question 1: both estimates, both MOEs,
$Z$, and the level. Read it with the cautions from Lab 4 in mind: a small group in a small
neighborhood has a large MOE, and a large MOE makes the test hard to pass. "Not significant"
on the Pacific Islander row is telling you about the sample size rather than about the people.

In [ ]:
race = add_standard_errors(race)     # Exercise #2 did this already; repeating it is harmless

rows = []
for g in GROUPS:
    col = f'pct_{g}'
    z_g = z_statistic(race, col, NBHD, COMP)
    rows.append({
        'Category': LABELS[g],
        f'{NBHD} (%)': race.loc[NBHD, col],
        f'{NBHD} MOE': race.loc[NBHD, col + '_moe'],
        f'{COMP} (%)': race.loc[COMP, col],
        f'{COMP} MOE': race.loc[COMP, col + '_moe'],
        'Z': z_g,
        'Significant?': significance(z_g),
    })

race_tests = pd.DataFrame(rows)
race_tests.round({f'{NBHD} (%)': 1, f'{NBHD} MOE': 1, f'{COMP} (%)': 1, f'{COMP} MOE': 1, 'Z': 2})

One caution before you write from that table. Running eight tests at once means that even if
nothing were really different, one of them would clear the 90 percent bar about one time in
ten just by chance. The table is still worth having. The discipline is to write about the
category your question is about, and to treat a lone marginal result on a small group with
some reserve.

In [ ]:
# EXERCISE #4: use z_statistic and significance on the Question 2 share (poverty rate or
# renter share) and confirm you get the same Z as Section 3. Then run the same two functions
# on the neighborhood versus the COUNTY row (the third row of q2). Does the conclusion change?
# One sentence as a comment on why the county comparison came out the way it did.

## 5. Writing it up

### 5.1 What "not significant" means, and what it does not

If $Z$ comes out below 1.645, the data cannot tell the difference from zero. Write that. Do
not write that the two places are the same, that the neighborhood "reflects" the city, that
the situation is stable, or that the result "speaks to" anything. A non-significant result
describes the evidence rather than the world: the true gap could be large in either
direction, and the survey is not big enough here to say. The honest sentence is short:

> The poverty rate in the neighborhood (18.2 percent, MOE 6.1) appears higher than in the
> county (11.4 percent, MOE 0.4), but the difference is not statistically significant at the
> 90 percent confidence level (Z = 1.5), so the data do not support a claim that the
> neighborhood's rate is higher.

And when the result **is** significant, what you have is a difference. "West Oakland has a
higher renter share than Oakland" is what the test supports. Why that is so is a different
question, and one these data do not answer.

### 5.2 What goes in the Data Note

Under every exhibit that makes a claim of difference or change:

* the source table and years, and the geography (which tracts, or city or county)
* how the neighborhood value was made (summed across N tracts, MOEs by root sum of squares;
  shares by the proportion formula; medians by the class rule, labeled an approximation)
* the test: "Differences tested with the two-estimate Z test (U.S. Census Bureau, 2020,
  Section 7), 90 percent confidence level" (or 95, or 99: the level you used)
* the result, if it is not already in the exhibit: "Z = 2.3, significant at 95 percent"

### 5.3 The numbers P/NP #5 asks for

P/NP #5 Part III wants, for each of your two comparisons, the estimate, the MOE, and the SE for
both geographies, then $Z$ and the level. The cell below prints all of it in the template's
order for both comparisons, so you can copy rather than retype. Set `RACE_COL` to the race or
ethnicity share you chose in Exercise #3.

In [ ]:
RACE_COL = 'pct_nh_black'      # the race or ethnicity share you chose for Comparison 1

def pnp5_block(df, col, label):
    z_val = z_statistic(df, col, NBHD, COMP)
    print(f'--- {label}: {col} ---')
    print(f'{"":28}{NBHD:>22}{COMP:>28}')
    print(f'{"Estimate (percent)":28}{df.loc[NBHD, col]:>22.1f}{df.loc[COMP, col]:>28.1f}')
    print(f'{"MOE (90 percent)":28}{df.loc[NBHD, col + "_moe"]:>22.2f}{df.loc[COMP, col + "_moe"]:>28.2f}')
    print(f'{"SE = MOE / 1.645":28}{df.loc[NBHD, col + "_se"]:>22.2f}{df.loc[COMP, col + "_se"]:>28.2f}')
    print(f'Difference: {abs(df.loc[NBHD, col] - df.loc[COMP, col]):.1f} percentage points')
    print(f'Z = |{df.loc[NBHD, col]:.1f} - {df.loc[COMP, col]:.1f}| / sqrt({df.loc[NBHD, col + "_se"]:.2f}^2 + {df.loc[COMP, col + "_se"]:.2f}^2) = {z_val:.2f}')
    print(f'Conclusion: {significance(z_val)}')
    print()

pnp5_block(race, RACE_COL, 'Comparison 1 (race and ethnicity)')
pnp5_block(q2, Q2_SHARE, 'Comparison 2 (economic conditions or housing)')

In [ ]:
# Save the tested tables. Download them with the notebook (right-click > Download).
race_tests.to_csv(f'lab5_{safe_name}_race_ethnicity_tests.csv', index=False)
q2.to_csv(f'lab5_{safe_name}_{Q2_OPTION}_with_se.csv')     # index kept: NAME is meaningful now
print('Saved.')

## 6. Part II: the same test across two periods

Assignment 1 Part II compares 2015 to 2019 with 2020 to 2024, and every claim of change must
be tested. The test is the one you just wrote. The only new thing is that the two rows are
the same place in two periods rather than two places in one period, so the row labels are
years instead of names. Two rules from the assignment carry over: compare only these two
periods (they do not share a year), and confirm each table's variable codes on the Groups
page for **both** vintages, because a few tables changed between them.

### 6.0 Your tract list is not the same in both periods. Check it.

Read this even if you are on the city or county path, because your Part I tracts are still
the ones you will describe.

Every ten years, with the decennial census, the Census Bureau redraws tract boundaries to keep
tracts near 4,000 residents: growing tracts are split, shrinking ones merged, and some are
renumbered or have a line moved. The 2015 to 2019 estimates are published on the **2010**
tracts and the 2020 to 2024 estimates on the **2020** tracts. So the list you pull for 2019 is
a different list from the one you pull for 2024, even when most of the numbers happen to
match. In Lab 4 and P/NP #4 you were given both lists for the field-trip neighborhoods; for
any other neighborhood you have to build the 2010 list yourself, and for some neighborhoods in
the country no clean 2010 list exists and a crosswalk is the only way.

The check has three steps, and the cell below does all three for whatever is in `TRACT_LIST`:

1. **Pull your 2020 numbers for 2019 and see what is missing.** A tract that was split,
   renumbered, or merged does not exist in the 2019 table under its 2020 number, and the API
   returns nothing for it without an error.
2. **For each 2020 tract, find the 2010 tract or tracts that covered the same land**, using
   the Census Bureau's relationship file (in this folder). Unchanged: same number. Split: one
   2010 tract, which you list once for 2019. Renumbered: one 2010 tract under a different
   number. More than one 2010 source with a real share of its land: a merge or a moved
   boundary.
3. **Decide.** If every tract is unchanged, split, or renumbered, the printed 2010 list is your
   `TRACT_LIST_2019` and Section 6.2 works as is. If any tract has more than one source, use
   the city or county for Part II, or open `Lab5_Crosswalk_Companion.ipynb`.

Whatever you find goes in your Data Notes.

No tracts, no boundary changes. Pull the same dictionary twice with `pull_geo`, once with
`year=2019` and once with `year=2024`, label the rows, stack them, add standard errors, and
test. The example is Oakland's renter share.

In [ ]:
STATE, COUNTY, PLACE = '06', '001', '53000'      # California, Alameda County, Oakland
TRACT_LIST = ['401400', '401500', '401600', '401700', '401800', '402200', '402400',
              '402500', '402600', '402700', '410500', '981900', '982000']     # your 2020 numbers, as in Lab 4

def check_tracts_2019(tract_list, state=STATE, county=COUNTY, relationship_file='tab20_tract20_tract10_st06.txt'):
    '''Report, for each 2020 tract, whether it exists in the 2015 to 2019 table and which 2010 tract(s) covered its land.

    Output: the 2010 tract list to use for 2019, or None if a tract needs the crosswalk companion.
    '''
    # Step 1: what does the 2019 table return for the 2020 numbers?
    df_2019 = pull_geo('tract:*', f'state:{state} county:{county}', {'NAME': 'NAME', 'B01001_001E': 'pop', 'B01001_001M': 'pop_moe'}, year=2019)
    found_2019 = set(df_2019['tract'])

    # Step 2: the 2010 tract(s) behind each 2020 tract, by share of the 2010 tract's land area
    cw = pd.read_csv(relationship_file, sep='|', dtype={'GEOID_TRACT_20': str, 'GEOID_TRACT_10': str}, encoding='utf-8-sig')
    cw['w10'] = cw['AREALAND_PART'] / cw['AREALAND_TRACT_10']
    cw = cw[cw['w10'] >= 0.02]          # ignore slivers under 2 percent of the 2010 tract's land

    list_2019, needs_crosswalk = [], False
    print(f'{"2020 tract":12}{"in 2019 table?":16}{"2010 source(s)":44}verdict')
    for t in tract_list:
        geoid20 = state + county + t
        sources = cw[cw['GEOID_TRACT_20'] == geoid20][['GEOID_TRACT_10', 'w10']]
        src = ', '.join(f'{g[5:]} ({w:.0%})' for g, w in sources.values)
        in_2019 = 'yes' if t in found_2019 else 'NO'
        if len(sources) == 1:
            g10, w = sources.iloc[0]
            t10 = g10[5:]
            # Did the rest of that 2010 tract go to other 2020 tracts? Then it was split.
            other_pieces = cw[(cw['GEOID_TRACT_10'] == g10) & (cw['GEOID_TRACT_20'] != geoid20)]
            if len(other_pieces) and other_pieces['w10'].max() >= 0.10:
                pieces = ', '.join(sorted(g[5:] for g in other_pieces['GEOID_TRACT_20']))
                verdict = f'split from {t10} (other pieces: {pieces}): use {t10} once for 2019'
            elif t10 == t:
                verdict = 'unchanged' if w > 0.98 else f'unchanged (edge adjusted; {1 - w:.0%} of its 2010 land moved to a neighbor)'
            else:
                verdict = f'renumbered: use {t10} for 2019'
            if t10 not in list_2019:
                list_2019.append(t10)
        else:
            verdict = 'more than one 2010 source: merged or boundary moved; city/county path or crosswalk companion'
            needs_crosswalk = True
        print(f'{t:12}{in_2019:16}{src:44}{verdict}')

    print()
    if needs_crosswalk:
        print('At least one tract has no single 2010 counterpart. TRACT_LIST_2019 cannot be built by hand.')
        return None
    print(f'2010 list for the 2019 pull ({len(list_2019)} tracts): {list_2019}')
    return list_2019

TRACT_LIST_2019 = check_tracts_2019(TRACT_LIST)

For the West Oakland default every row says unchanged and the two lists are identical, which
is why the field-trip neighborhoods were a gentle introduction. Run the same cell on your own
tracts before you pull anything for 2019, and copy the verdicts into your Data Notes. (The
check ignores slivers under 2 percent of a 2010 tract's land, which the Bureau creates when it
moves a line to a street centerline; those do not change who lives where.)

### 6.1 New learner path: the city or county in both periods

In [ ]:
city_2019 = add_shares(pull_geo(f'place:{PLACE}', f'state:{STATE}', tenure_vars, year=2019), ['renter'], total='units')
city_2024 = add_shares(pull_geo(f'place:{PLACE}', f'state:{STATE}', tenure_vars, year=2024), ['renter'], total='units')

city_2019['period'] = '2015 to 2019'
city_2024['period'] = '2020 to 2024'

city_change = pd.concat([city_2019, city_2024], ignore_index=True).set_index('period')
city_change = add_standard_errors(city_change)

z_city = z_statistic(city_change, 'pct_renter', '2015 to 2019', '2020 to 2024')
print(f'Z = {z_city:.2f}: {significance(z_city)}')
city_change[['NAME', 'units', 'renter', 'pct_renter', 'pct_renter_moe', 'pct_renter_se']].round(2)

### 6.2 Advanced path: the neighborhood in both periods

Same idea, with the Lab 4 Section 8.2 pattern for the 2019 pull, using the two lists from
Section 6.0: `TRACT_LIST` (2020 numbers) for 2024 and `TRACT_LIST_2019` (2010 numbers) for
2019. A split tract is listed once for 2019 and as its pieces for 2024; the pieces add back up
to the same land. A renumbered tract is listed under its 2010 number for 2019. If Section 6.0
returned `None`, one of your tracts has no single 2010 counterpart, and this section will not
line up: use the city or county for Part II, or open `Lab5_Crosswalk_Companion.ipynb`.

In [ ]:
# TRACT_LIST (2020 numbers) and TRACT_LIST_2019 (2010 numbers) come from Section 6.0.
# If the check returned None, stop here: this section is not for your tracts.
assert TRACT_LIST_2019 is not None, 'Section 6.0 found a tract with more than one 2010 source. Use the city/county path or the crosswalk companion.'

def neighborhood_period(tract_list, year, variables, groups, total, name):
    '''Pull the county's tracts for one year, keep tract_list, aggregate, and add shares.'''
    alltr = pull_geo('tract:*', f'state:{STATE} county:{COUNTY}', variables, year=year)
    tracts = alltr[alltr['tract'].isin(tract_list)].copy()
    missing = sorted(set(tract_list) - set(tracts['tract']))
    print(f'{year}: {len(tracts)} of {len(tract_list)} tracts found.', 'Not found: ' + str(missing) if missing else '')
    return add_shares(aggregate_tracts(tracts, name), groups, total=total)

nbhd_2019 = neighborhood_period(TRACT_LIST_2019, 2019, tenure_vars, ['renter'], 'units', NEIGHBORHOOD_NAME)
nbhd_2024 = neighborhood_period(TRACT_LIST, 2024, tenure_vars, ['renter'], 'units', NEIGHBORHOOD_NAME)
nbhd_2019['period'] = '2015 to 2019'
nbhd_2024['period'] = '2020 to 2024'

nbhd_change = add_standard_errors(pd.concat([nbhd_2019, nbhd_2024], ignore_index=True).set_index('period'))
z_nbhd = z_statistic(nbhd_change, 'pct_renter', '2015 to 2019', '2020 to 2024')
print(f'Z = {z_nbhd:.2f}: {significance(z_nbhd)}')
nbhd_change[['NAME', 'n_tracts', 'units', 'renter', 'pct_renter', 'pct_renter_moe', 'pct_renter_se']].round(2)

Compare the two $Z$ values. The city's margins of error are small, so a modest change can be
significant; the neighborhood's are large, so the same size of change may not be. That is the
reason the assignment offers the city or county path to new learners: the arithmetic is the
same, and the boundaries hold still.

In [ ]:
# EXERCISE #5: run a change-over-time test on ONE variable from the Part II track you chose,
# for your own comparison geography (new learner path, Section 6.1 pattern) or your own
# tracts (advanced path, Section 6.2 pattern). Write a new dictionary if your table is not
# in the toolkit (confirm the codes on the 2019 AND 2024 Groups pages). Print Z and the level,
# and write the sentence for your memo as a comment, in the form of Section 3.3.

### 6.3 Adjusting dollars for inflation

ACS dollar figures are reported in the dollars of each period's final year: the 2015–2019 estimates are in 2019 dollars, and the 2020–2024 estimates are in 2024 dollars. To compare them, convert the 2019 values to 2024 dollars using the Consumer Price Index for All Urban Consumers (CPI-U), published by the U.S. Bureau of Labor Statistics.

Factor = CPI-U 2024 annual average ÷ CPI-U 2019 annual average = 313.689 ÷ 255.657 = 1.227

Multiply a 2019 dollar estimate and its MOE by 1.227. Use the annual averages, not a single month: ACS dollars cover a full year.

Official source (cite this). U.S. Bureau of Labor Statistics, Consumer Price Index for All Urban Consumers (CPI-U), U.S. city average, all items, series CUUR0000SA0. https://www.bls.gov/cpi/

To pull the table yourself:

1. Type the series ID CUUR0000SA0 into the search bar on bls.gov and open the result.
2. The table opens showing monthly values and two half-year averages, not the annual average. Select annual averages in the display options, along with the years you want.
3. Click Retrieve data. The annual average column is the one to use.

Easier to read. usinflationcalculator.com republishes the same BLS figures in one table, with monthly values and the annual average for every year since 1913 (use the "Avg" column): https://www.usinflationcalculator.com/inflation/consumer-price-index-and-annual-percent-changes-from-1913-to-2008/ It's a private site, so cite BLS, not this page.

In your Data Note, say which you did: "2015–2019 values adjusted to 2024 dollars using the CPI-U annual average (BLS series CUUR0000SA0; factor 1.227)," or "values in nominal dollars, not adjusted for inflation."

In [ ]:
CPI_FACTOR = 1.227      # CPI-U annual average 2024 / 2019 = 313.689 / 255.657 (BLS series CUUR0000SA0)

# Median gross rent (B25064) for Oakland in both periods. A city's median is published with its
# own MOE, so the test is on published numbers here, not on the class-rule approximation.
rent_2019 = pull_geo(f'place:{PLACE}', f'state:{STATE}', rent_vars, year=2019)
rent_2024 = pull_geo(f'place:{PLACE}', f'state:{STATE}', rent_vars, year=2024)

# Adjust the 2019 estimate AND its margin of error. Both are in 2019 dollars; both scale.
rent_2019_adj = rent_2019.copy()
rent_2019_adj['med_rent'] = rent_2019['med_rent'] * CPI_FACTOR
rent_2019_adj['med_rent_moe'] = rent_2019['med_rent_moe'] * CPI_FACTOR

rent_2019['period'] = '2015 to 2019 (2019 dollars, nominal)'
rent_2019_adj['period'] = '2015 to 2019 (in 2024 dollars)'
rent_2024['period'] = '2020 to 2024 (2024 dollars)'

rent_change = add_standard_errors(pd.concat([rent_2019, rent_2019_adj, rent_2024], ignore_index=True).set_index('period'))

z_nominal = z_statistic(rent_change, 'med_rent', '2015 to 2019 (2019 dollars, nominal)', '2020 to 2024 (2024 dollars)')
z_real = z_statistic(rent_change, 'med_rent', '2015 to 2019 (in 2024 dollars)', '2020 to 2024 (2024 dollars)')
print(f'Nominal comparison: Z = {z_nominal:.2f}, {significance(z_nominal)}')
print(f'Constant-dollar comparison: Z = {z_real:.2f}, {significance(z_real)}')
rent_change[['NAME', 'med_rent', 'med_rent_moe', 'med_rent_se']].round(0)

Read the two $Z$ values against each other. The nominal comparison mixes a real change in
rents with five years of general inflation, so it tends to find a "change" that is partly the
dollar shrinking. The constant-dollar comparison is the one that answers "did rents rise
faster than prices in general?" Either is allowed in the memo, as long as the Data Note says
which one you did.

In [ ]:
# EXERCISE #6: pick one dollar variable (median household income B19013, median gross rent
# B25064, or median house value B25077) for your own city or county. Pull both periods, adjust
# the 2019 estimate and MOE with CPI_FACTOR, test 2019-adjusted against 2024, and print Z and
# the level. Then write the Data Note sentence as a comment (use the wording in Section 6.3).

## 7. If you want more: a bar chart with error bars

Lab 6 (October 7 and 9) is about charts, so this is only a preview. A chart that shows
margins of error lets a reader see the test: if the error bars of two bars overlap heavily,
the difference is unlikely to be significant. (Overlap is a rough guide, not the test itself;
two intervals can overlap a little and still be significantly different.)

`df.plot.bar()` draws a bar per row; `yerr=` names the column to use for the error bars. Our
percent columns already run from 0 to 100, so the axis reads in percent as is.

In [ ]:
import matplotlib.pyplot as plt

ax = q2.loc[[NBHD, COMP]].plot.bar(
    y=Q2_SHARE,
    yerr=f'{Q2_SHARE}_moe',
    legend=False,
    capsize=4,
    color='#1B4A82',
    figsize=(5, 4),
)
ax.set_ylabel('Percent')
ax.set_xlabel('')
ax.set_title(f'{Q2_SHARE.replace("pct_", "").replace("_", " ").title()}, 2020 to 2024 (error bars: 90 percent MOE)')
plt.xticks(rotation=0)
plt.tight_layout()

A chart for the memo also needs a note under it: source table and years, geography, what the
error bars show, and the test result. That is Lab 6.

## 8. What to do now

### For P/NP #5 (due Sunday, October 4)

Part III needs two tests, both on your own tracts and your own comparison geography:

1. Comparison 1: one race or ethnicity share (Section 3 by hand, or Section 4 with the
   function; the printout in Section 5.3 is in the template's order)
2. Comparison 2: the poverty rate (Option A) or the renter share (Option B), the same way
3. For each: the estimate, MOE, and SE for both geographies, $Z$, the confidence level, and one
   or two sentences for a non-specialist reader (Section 3.3 for a significant result,
   Section 5.1 for one that is not)

If you ran today's lab on the West Oakland default, swap in your own Lab 4 CSVs (Section 1.1)
and run the notebook again; everything downstream follows.

### For Assignment 1 (due Sunday, October 11)

* **Part I:** test every difference you claim between the neighborhood and the city or county.
  Section 4.1 gives you the whole race and ethnicity table at once; the same loop works for
  any set of shares (commute modes, income bands, tenure by race).
* **Part II:** run the Section 6.0 check on your tracts first; the 2019 list is a 2010 list
  and it is usually not identical to your 2020 list. Then test every change you claim between
  2015 to 2019 and 2020 to 2024, on the path you chose (Sections 6.1 and 6.2). Adjust dollars
  or say you did not (Section 6.3). If the check returned `None`, open the crosswalk companion
  or keep Part II at the city or county scale.
* **Data Notes:** Section 5.2 is the checklist. The test and its level go in the note under
  every exhibit that makes a claim.
* **Medians:** the class-rule neighborhood median is an approximation and its MOE is not
  exact, so do not run this test on it. Test the published city or county median across
  periods (Section 6.3), report the tract range, and describe the neighborhood median in
  words.

The rest of today's lab is open work time on Assignment 1. The GSIs are here for exactly the
question you are stuck on.

## Before you leave

* You can turn any MOE into an SE, and any two estimates into a $Z$.
* You have `z_statistic()` and `significance()`, and you have used them on three comparisons.
* You know what to write when the result is not significant.
* You have checked your tracts in both periods and know which 2010 numbers to pull for 2019.
* Your CSVs and this notebook are downloaded, not just saved in Datahub.

**Coming up:** Monday October 5 is on visual displays of quantitative data; Lab 6 builds the
charts for your memo. **P/NP #5 is due Sunday, October 4. Assignment 1 is due Sunday,
October 11.**